# Limpeza de Dados e Feature Engineering
**Objetivo:** Transformar os dados brutos da ANEEL em um dataset tabular pronto para os modelos de Machine Learning, removendo colunas que causam *Data Leakage* (vazamento do futuro) e criando a variável alvo.

In [1]:
# Importações
import pandas as pd
import numpy as np
import os

# Configuração de caminhos
RAW_PATH = '../data/raw/dados_aneel.xlsx' 
PROCESSED_PATH = '../data/processed/dataset_limpo.csv'

In [4]:
# Carregamento de dados
df = pd.read_excel(RAW_PATH)

print(f"Total de linhas iniciais: {df.shape[0]}")
print(f"Total de colunas iniciais: {df.shape[1]}")
df.head()

Total de linhas iniciais: 65231
Total de colunas iniciais: 14


,Distribuidora,Protocolo,data_criacao,data_encerramento,Natureza,Tipologia,CanalEntrada,Forma_Encerramento,DECISAO,Qtd_dias_na_Concessionario,PercentualConcessionario,Qtd_dias_na_Agencia,PercentualAgencia,Prazo_Medio_Tratamento
0,ENEL CE,103365732169,2021-03-03 10:50:39.067,2026-03-24 10:52:06.600,RECLAMAÇÃO,Cobrança Indevida,Fale Conosco,Processo Finalizado,PROCEDENTE,33,1.79,1814,98.21,1847.0
1,CEMIG,104752952314,2023-08-17 16:50:39.870,2026-03-27 15:09:21.683,RECLAMAÇÃO,Conexão de Mini e Microgeração,Fale Conosco,Encerrada,PROCEDENTE,948,99.48,5,0.52,953.0
2,CEMIG,104765912332,2023-08-29 15:03:12.853,2026-02-25 11:22:53.613,RECLAMAÇÃO,Alteração de carga,Fale Conosco,Encerrada,IMPROCEDENTE,900,98.79,11,1.21,911.0
3,ENEL RJ,104869582371,2023-11-16 13:18:41.140,2026-01-23 17:18:30.863,RECLAMAÇÃO,Extensão de Rede,Fale Conosco,Encerrada,PROCEDENTE,765,95.74,34,4.26,799.0
4,CEMIG,104892392311,2023-11-29 12:47:43.943,2026-01-08 15:49:00.567,RECLAMAÇÃO,Extensão de Rede,Fale Conosco,Encerrada,PROCEDENTE,753,97.67,18,2.33,771.0


In [5]:
# 1. Garantir que as colunas de data são reconhecidas como datetime
df['data_criacao'] = pd.to_datetime(df['data_criacao'])
df['data_encerramento'] = pd.to_datetime(df['data_encerramento'])

# 2. Criar Target Regressão (Tempo de Resolução em Dias)
# Adicionamos uma pequena margem (ex: dt.total_seconds() / 86400) ou apenas dias inteiros
df['tempo_resolucao_dias'] = (df['data_encerramento'] - df['data_criacao']).dt.total_seconds() / 86400

# 3. Criar Target Classificação (Estourou SLA?)
# Exemplo: Vamos assumir que o SLA da ANEEL seja 30 dias. 
SLA_DIAS = 30
df['estourou_sla'] = np.where(df['tempo_resolucao_dias'] > SLA_DIAS, 1, 0)

print(f"Distribuição do estouro de SLA (0 = No prazo, 1 = Atrasado):\n{df['estourou_sla'].value_counts(normalize=True) * 100}")

Distribuição do estouro de SLA (0 = No prazo, 1 = Atrasado):
estourou_sla
0    95.741289
1     4.258711
Name: proportion, dtype: float64


In [6]:
# Como não podemos dar a data crua para o modelo, extraímos características temporais úteis
df['dia_semana_abertura'] = df['data_criacao'].dt.dayofweek # 0=Segunda, 6=Domingo
df['hora_abertura'] = df['data_criacao'].dt.hour
df['mes_abertura'] = df['data_criacao'].dt.month

df[['data_criacao', 'dia_semana_abertura', 'hora_abertura', 'mes_abertura']].head()

,data_criacao,dia_semana_abertura,hora_abertura,mes_abertura
0,2021-03-03 10:50:39.067,2,10,3
1,2023-08-17 16:50:39.870,3,16,8
2,2023-08-29 15:03:12.853,1,15,8
3,2023-11-16 13:18:41.140,3,13,11
4,2023-11-29 12:47:43.943,2,12,11


In [7]:
# Colunas que a máquina não deve ver pois só existem APÓS o encerramento, evitamos data leakage
colunas_futuro = [
    'data_criacao', # Já extraímos o que precisava dela
    'data_encerramento', 
    'Forma_Encerramento', 
    'DECISAO', 
    'Qtd_dias_na_Concessionario', 
    'PercentualConcessionario',
    'Qtd_dias_na_Agencia',
    'PercentualAgencia',
    'Prazo_Medio_Tratamento'
]

# Dropando as colunas
df_limpo = df.drop(columns=colunas_futuro, errors='ignore') # errors='ignore' evita erro se você errar o nome de uma coluna

print("Colunas finais que o modelo vai enxergar (Features + Targets):")
print(df_limpo.columns.tolist())

Colunas finais que o modelo vai enxergar (Features + Targets):
['Distribuidora', 'Protocolo', 'Natureza', 'Tipologia', 'CanalEntrada', 'tempo_resolucao_dias', 'estourou_sla', 'dia_semana_abertura', 'hora_abertura', 'mes_abertura']


In [8]:
# Remover linhas onde o alvo calculou errado (ex: data encerramento menor que a de criação)
df_limpo = df_limpo[df_limpo['tempo_resolucao_dias'] >= 0]

# Salvar o arquivo final processado
df_limpo.to_csv(PROCESSED_PATH, index=False)

print(f"Dataset limpo salvo com sucesso em: {PROCESSED_PATH}")
print(f"Tamanho final: {df_limpo.shape}")

Dataset limpo salvo com sucesso em: ../data/processed/dataset_limpo.csv
Tamanho final: (65231, 10)
